In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
font = {'size' : 15}
matplotlib.rc('font', **font)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Helvetica"
})
import numpy.fft as fft
from numba import njit
import os

In [ ]:
source_folder = "interface_data/"
os.makedirs("plots", exist_ok=True)

In [ ]:
fields = [0.13, 0.14, 0.15, 0.16]
fields_colors = matplotlib.colormaps.get('tab10')(range(4))

### $S(q)$

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
for color, m_field in zip(fields_colors, fields):
    interface = np.load("%sinterface_field=%1.2f.npz" % (source_folder,m_field))['interface']
    q = fft.fftfreq(interface.shape[1])
    L = interface.shape[1]
    if(m_field == 0.13):
        h_q = fft.fft((interface-interface.mean(axis=1)[:,None])[100:,:], axis=1)
    else:
        h_q = fft.fft((interface-interface.mean(axis=1)[:,None])[-40000:,:], axis=1)
    s_q = h_q*np.conj(h_q)
    plt.scatter(2*np.pi*q[q>0],s_q.real.mean(axis=0)[q>0], s=5, color=color, label='$%1.2f$ mT' % m_field)
plt.plot(2*np.pi*q[q>0],5/q[q>0]**2.26,ls='dashed', c='black', lw=2) #, label="qKPZ")
plt.plot(2*np.pi*q[q>0],5/q[q>0]**2.33,ls='dashed', c='grey', lw=2) #, label="qEW")
plt.plot(2*np.pi*q[q>0],5/q[q>0]**3.50,ls='-.', c='grey', lw=2) #label="equilibrium")
plt.xscale('log')
plt.yscale('log')
plt.xlabel("$q \\quad (\\rm{rad^{-1}})$", fontsize=20)
plt.ylabel("$S(q)$", fontsize=20)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.ylim(plt.gca().get_ylim()[0], 1e9)
lgnd = plt.legend(fontsize=15, loc='lower left')
for lh in lgnd.legend_handles:
    lh._sizes = [36]
plt.tight_layout()
plt.savefig("plots/sq.pdf", bbox_inches='tight')
plt.show()

### $N(\tau)/L$

In [ ]:
@njit
def total_change(data, t_0, tau_max):
    changes = np.zeros(tau_max)
    for i, t_i in enumerate(range(t_0+1, t_0+tau_max+1)):
        changes[i] =(data[t_i, :] > data[t_0,:]).sum()/data.shape[1]
    return changes

In [ ]:
field = 0.13
data = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))['interface']
tau_max = 2000
curves = []
for t0 in range(0, data.shape[0]-tau_max):
    curves.append(total_change(data, t0, tau_max))
curves = np.array(curves)

In [ ]:
n1,n2,n3 = curves[:30000].mean(axis=0), curves[30000:].mean(axis=0), curves[:].mean(axis=0)
t1,t2,t3 = np.argsort(np.abs(n1-0.5))[:10].mean(),np.argsort(np.abs(n2-0.5))[:10].mean(), np.argsort(np.abs(n3-0.5))[:10].mean()
plt.plot(np.arange(1, 1+tau_max),n1,color='red', label="Initial regime; $\\tau^*=%i$" % t1)
plt.plot(np.arange(1, 1+tau_max),n2,color='blue',  label="Stationary regime; $\\tau^*=%i$" % t2)
plt.plot(np.arange(1, 1+tau_max),n3,color='black', ls='dashed',  label="Global average; $\\tau^*=%i$" % t3)
plt.legend(fontsize=15)
plt.xlabel("$\\tau$", fontsize=15)
plt.ylabel("$N(\\tau)/L$", fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.savefig("plots/Nt_field=%1.2f.pdf" % field, bbox_inches='tight')
plt.show()